# ML-2 Supervised Learning
# Linear regression model

## 1. Answer the questions

Постановка задачи:
- $y = (y_1, y_2, ..., y_N)$ - Вектор целевых значений, таргет  
- $X = (x_{ij}) \in \mathbb{R^{N\times d}}$ - Матрица признаков ($j$-ый столбец соответствует $j$-ому признаку, $i$-ая строка соответствует вектору-признаков $i$-ого объекта, $x_{ij}$ - значение $j$-го признака для $i$-го объекта)

Ищем именно линейную зависимость таргета от признаков. То есть ищем приближающую функцию среди функций вида:  
$$f_w(x_i) = \langle x_i, w\rangle = \sum_{j=1}^d x_{ij} w_j$$
  
Приближающую к таргету - значит минимизирующую какое-то различие, то есть какую-то функцию потерь. 
Рассмотрим в качестве функции потерь **MSE (Mean Squared Error)**:
$$MSE(f_w, X, y) = \frac{1}{N} \| y - Xw \|_2^2$$

где $\| \cdot \|_2$ — евклидова норма ($L^2$-норма), определяемая как:
$$\|z\|_2 = \sqrt{\sum_{i=1}^{N} z_i^2}$$
Получаем задачу оптимизации:
$$\frac{1}{N}\|y - Xw\|_2^2 \rightarrow \min_{w \in \mathbb{R}^d}$$
отсюда понятен вектор $w$, который будет одназначно задавать нашу искомую функцию $f_w(x_i)$ :
$$
w = \arg\min_{w \in \mathbb{R}^d} \frac{1}{N}\|y - Xw\|_2^2
$$


Решим геометрически: 
$x^{(1)}, \dots, x^{(d)}$ - столбцы матрицы $X$, то есть столбцы признаков. Тогда:
$$
Xw = w_1x^{(1)} + \dots + w_dx^{(d)}
$$
и задачу регрессии можем переформулировать: найти линейную комбинацию столбцов $x^{(1)}, \dots, x^{(d)}$, которая лучше всего приблежает $y$ по евклидовой норме, то есть ищем проекцию y на подпространство $\langle x^{(1)}, \dots, x^{(d)}\rangle$  

Разложим $y = y_\parallel + y_\perp$, где $y_\parallel = Xw$ — та самая проекция, а $y_\perp = y - Xw$ — ортогональная составляющая, то есть: 
$$
y_\perp \perp \langle x^{(1)}, \dots, x^{(D)} \rangle
$$

Выразим это в матричном виде:

$$X^T (y - Xw) = 0.$$

В самом деле, каждый элемент столбца $X^T (y - Xw)$ — это скалярное произведение строки $X^T$ (столбца $X =$ одного из $x^{(i)}$) на $y - Xw$. Из уравнения $X^T (y - Xw) = 0$ выражаем $w$:

$$w = (X^T X)^{-1} X^T y$$

### ii. Добавление $L_1, L_2$ регуляризации в функцию потерь

Lasso/L1 regularization:
$$
w^* = \arg\min_{w} (\frac{1}{n}||Y-Xw||_2^2 + \lambda ||w||_1), \quad \lambda > 0 
$$

$$
\frac{2}{n} X^\top (Y - Xw) - \lambda sgn(w) = 0
$$
---

Tikhonov/$\mathcal{l}2$/ridge regularization instead solves the **penalized** problem:
$$
w^* = \arg\min_{w} (\mathbb{E}[(Y-w^\top X)^2] + \lambda ||w||_2^2), \quad \lambda > 0 
$$
$$
(X^\top X + \lambda I)^{-1} X^\top Y
$$

## iii. Почему $L_1$ часто используют для выбора признаков? 

- **L1-регуляризация** заставляет веса становиться равными нулю за счёт вычитания константы $ \lambda \cdot \text{sgn}(w_i)\ $ при обновлении градиента.
- В центрированных данных левая часть уравнения — это **ковариация** признака с остатками модели.
- Если признак не влияет на предсказание, ковариация близка к нулю, и вес **обнуляется**.
- В результате признак **исключается** из модели — это и есть отбор признаков.

### iv. Как с помощью линейных моделей можно подобрать нелинейные зависимости? 

- Использовать **базисные функции**: добавить полиномиальные признаки $ (x^2, x^3) $, тригонометрические или другие нелинейные преобразования исходных признаков.
- Применить **ядровой трюк** (kernel trick) — линейная модель в пространстве признаков высокой размерности соответствует нелинейной в исходном.
- Использовать **смешанные модели**: линейная комбинация нелинейных базисных функций сохраняет интерпретируемость и преимущества линейной оптимизации.

## 2. Introduction

### i. Import libraries.

In [224]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.model_selection import train_test_split

## ii. Read Train and Test Parts

In [225]:
df = pd.read_json(path_or_buf="../../datasets/train.json")
df = df[
    df["price"].between(
        left=df["price"].quantile(q=0.01),
        right=df["price"].quantile(q=0.99),
        inclusive="both",
    )
]

y = df["price"]
X = df[['bathrooms', 'bedrooms']]

## 3. Intro data analysis part 2

### ii. Remove unused symbols ([,], ', ", and space) from the column

In [226]:
def clean_features(feature_column):
    """
    очищает колонку features от лишних символов:
    """
    cleaned = []
    for item in feature_column:
        if isinstance(item, str):
            clean_str = item.strip('[]').replace("'", "").replace('"', '').replace(' ', '')
            parts = [p.strip() for p in clean_str.split(',') if p.strip()]
            cleaned.append(parts)
        else:
            cleaned.append(item)
    return cleaned

df['features_cleaned'] = clean_features(df['features'])

print("После очистки:")
print(df[['features', 'features_cleaned']].head(3))

После очистки:
                                            features  \
4  [Dining Room, Pre-War, Laundry in Building, Di...   
6  [Doorman, Elevator, Laundry in Building, Dishw...   
9  [Doorman, Elevator, Laundry in Building, Laund...   

                                    features_cleaned  
4  [Dining Room, Pre-War, Laundry in Building, Di...  
6  [Doorman, Elevator, Laundry in Building, Dishw...  
9  [Doorman, Elevator, Laundry in Building, Laund...  


### iii. Collect all feature values into one huge list

In [227]:
all_features = []

for idx, row in df.iterrows():
    # Расширяем список значениями из features_cleaned
    all_features.extend(row['features_cleaned'])

print(f"Длина всего списка: {len(all_features)}")
print("\nПервые 20 значений:")
for i, val in enumerate(all_features[:20], 1):
    print(f"{i:2}. {val}")

Длина всего списка: 262573

Первые 20 значений:
 1. Dining Room
 2. Pre-War
 3. Laundry in Building
 4. Dishwasher
 5. Hardwood Floors
 6. Dogs Allowed
 7. Cats Allowed
 8. Doorman
 9. Elevator
10. Laundry in Building
11. Dishwasher
12. Hardwood Floors
13. No Fee
14. Doorman
15. Elevator
16. Laundry in Building
17. Laundry in Unit
18. Dishwasher
19. Hardwood Floors
20. Doorman


### iv. How many unique values does a result list contain?

In [228]:
unique_features = set(all_features)
print(f"Количество уникальных значений: {len(unique_features)}")

Количество уникальных значений: 1539


### v. Quantity statistics about data

In [229]:
feature_counter = Counter(all_features)

print("Первые 15 самых частых признаков:")
for i, (feature, count) in enumerate(feature_counter.most_common(15), 1):
    print(f"{i:2}. {feature:30} — {count}")

Первые 15 самых частых признаков:
 1. Elevator                       — 25398
 2. Hardwood Floors                — 23159
 3. Cats Allowed                   — 23148
 4. Dogs Allowed                   — 21662
 5. Doorman                        — 20497
 6. Dishwasher                     — 20095
 7. No Fee                         — 17806
 8. Laundry in Building            — 16093
 9. Fitness Center                 — 13000
10. Pre-War                        — 8978
11. Laundry in Unit                — 8448
12. Roof Deck                      — 6423
13. Outdoor Space                  — 5137
14. Dining Room                    — 4901
15. High Speed Internet            — 4225


### vi. Count the most popular functions from our huge list and take the top 20 for this moment.

In [230]:
top_20 = feature_counter.most_common(20)
top_feature_names = [feature for feature, _ in top_20]

print(f"\nСписок названий топ-20:")
for i, name in enumerate(top_feature_names, 1):
    print(f"{i:2}. {name}")


Список названий топ-20:
 1. Elevator
 2. Hardwood Floors
 3. Cats Allowed
 4. Dogs Allowed
 5. Doorman
 6. Dishwasher
 7. No Fee
 8. Laundry in Building
 9. Fitness Center
10. Pre-War
11. Laundry in Unit
12. Roof Deck
13. Outdoor Space
14. Dining Room
15. High Speed Internet
16. Balcony
17. Swimming Pool
18. Laundry In Building
19. New Construction
20. Terrace


### viii. Create 20 new features based on the top 20 values: 1 if the value is in the "Feature" column, otherwise 0

In [231]:
df_expanded = df.copy()

for feature_name in top_feature_names:
    df_expanded[feature_name] = df_expanded['features_cleaned'].apply(
        lambda x: 1 if feature_name in x else 0
    )

print("Новые признаки созданы")
print(f"\nПервые 5 строк новых признаков:")
print(df_expanded[top_feature_names[:5]].head())

print(f"\nСумма значений по каждому новому признаку (сколько раз встречается):")
for feature_name in top_feature_names[:5]:
    print(f"  {feature_name}: {df_expanded[feature_name].sum()}")

Новые признаки созданы

Первые 5 строк новых признаков:
    Elevator  Hardwood Floors  Cats Allowed  Dogs Allowed  Doorman
4          0                1             1             1        0
6          1                1             0             0        1
9          1                1             0             0        1
10         0                0             0             0        0
15         1                0             0             0        1

Сумма значений по каждому новому признаку (сколько раз встречается):
  Elevator: 25398
  Hardwood Floors: 23159
  Cats Allowed: 23148
  Dogs Allowed: 21662
  Doorman: 20497


### ix. Create a special variable feature_list with all feature names. Now we have 22 values. All models should be trained on these 22 features.

In [232]:
base_features = ['bathrooms', 'bedrooms']

feature_list = base_features + top_feature_names

print(f"Всего признаков: {len(feature_list)}")
print("\nСписок всех признаков:")
for i, f in enumerate(feature_list, 1):
    print(f"{i:2}. {f}")

X = df_expanded[feature_list]
y = df['price']

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

Всего признаков: 22

Список всех признаков:
 1. bathrooms
 2. bedrooms
 3. Elevator
 4. Hardwood Floors
 5. Cats Allowed
 6. Dogs Allowed
 7. Doorman
 8. Dishwasher
 9. No Fee
10. Laundry in Building
11. Fitness Center
12. Pre-War
13. Laundry in Unit
14. Roof Deck
15. Outdoor Space
16. Dining Room
17. High Speed Internet
18. Balcony
19. Swimming Pool
20. Laundry In Building
21. New Construction
22. Terrace

X shape: (48379, 22)
y shape: (48379,)


## 4. Models implementation — Linear regression

### i. Implement a Python class for a linear regression algorithm

In [233]:
class MyLinearRegressionAnalytical:
    """
    Linear Regression using Analytical Solution (Normal Equations)
    """
    
    def __init__(self):
        self.weights = None
        
    def fit(self, X, y):
        """
        Fit the model using analytical solution.
        
        X: numpy array (n_samples, n_features)
        y: numpy array (n_samples,)
        """
        X_with_bias = np.c_[np.ones(X.shape[0]), X]
        
        self.weights = np.linalg.pinv(X_with_bias.T @ X_with_bias) @ X_with_bias.T @ y
        
        return self
    
    def predict(self, X):
        """
        Predict using the linear model.
        
        """
        X_with_bias = np.c_[np.ones(X.shape[0]), X]
        return X_with_bias @ self.weights

print("Класс MyLinearRegressionAnalytical создан")

Класс MyLinearRegressionAnalytical создан


### ii. What is determenistic model? Make SGD determenistic.

### iii. R² coefficient (coefficient of determination)

In [234]:
def my_r2_score(y_true, y_pred):
    """
    Calculate R² coefficient (coefficient of determination)
    """
    y_mean = np.mean(y_true)
    
    ss_res = np.sum((y_true - y_pred) ** 2)
    
    ss_tot = np.sum((y_true - y_mean) ** 2)
    
    if ss_tot == 0:
        return 1.0
    
    r2 = 1 - (ss_res / ss_tot)
    
    return r2

print("Проверка работы функции:")
y_true_test = np.array([1, 2, 3, 4, 5])
y_pred_perfect = np.array([1, 2, 3, 4, 5])
y_pred_bad = np.array([3, 3, 3, 3, 3])

print(f"Идеальное предсказание: R² = {my_r2_score(y_true_test, y_pred_perfect):.4f}")
print(f"Предсказание средним: R² = {my_r2_score(y_true_test, y_pred_bad):.4f}")

Проверка работы функции:
Идеальное предсказание: R² = 1.0000
Предсказание средним: R² = 0.0000


In [235]:
def print_results_table(model_name, mae_train, mae_test, rmse_train, rmse_test, r2_train, r2_test):
    """
    Выводит результаты модели в виде таблицы
    """
    data = {
        'Metric': ['MAE', 'RMSE', 'R²'],
        'Train': [f"{mae_train:.2f}", f"{rmse_train:.2f}", f"{r2_train:.4f}"],
        'Test': [f"{mae_test:.2f}", f"{rmse_test:.2f}", f"{r2_test:.4f}"]
    }
    df = pd.DataFrame(data)
    print(f"{model_name}:")
    display(df)

def create_summary_table(models_data, title="SUMMARY TABLE"):
    """
    Создает сводную таблицу для всех моделей
    """
    results = pd.DataFrame(
        np.nan,
        index=pd.MultiIndex.from_product(
            [[name for name, _, _, _, _, _, _ in models_data], ["train", "test"]],
            names=["model", "split"],
        ),
        columns=["MAE", "RMSE", "R2"],
    )
    
    for name, mae_tr, mae_te, rmse_tr, rmse_te, r2_tr, r2_te in models_data:
        results.loc[(name, "train"), "MAE"] = mae_tr
        results.loc[(name, "train"), "RMSE"] = rmse_tr
        results.loc[(name, "train"), "R2"] = r2_tr
        results.loc[(name, "test"), "MAE"] = mae_te
        results.loc[(name, "test"), "RMSE"] = rmse_te
        results.loc[(name, "test"), "R2"] = r2_te
    
    for col in ["MAE", "RMSE"]:
        results[col] = results[col].map(lambda x: f"{x:.2f}")
    results["R2"] = results["R2"].map(lambda x: f"{x:.4f}")
    
    print(title)
    display(results)

#### iv. Train custom model and evaluate

In [236]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21
)

In [237]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [238]:
my_model = MyLinearRegressionAnalytical()
my_model.fit(X_train_scaled, y_train)

y_train_pred_my = my_model.predict(X_train_scaled)
y_test_pred_my = my_model.predict(X_test_scaled)

mae_train_my = np.mean(np.abs(y_train - y_train_pred_my))
mae_test_my = np.mean(np.abs(y_test - y_test_pred_my))

rmse_train_my = np.sqrt(np.mean((y_train - y_train_pred_my) ** 2))
rmse_test_my = np.sqrt(np.mean((y_test - y_test_pred_my) ** 2))

r2_train_my = my_r2_score(y_train, y_train_pred_my)
r2_test_my = my_r2_score(y_test, y_test_pred_my)

print_results_table(
    "MY LINEAR REGRESSION:",
    mae_train_my, mae_test_my,
    rmse_train_my, rmse_test_my,
    r2_train_my, r2_test_my
)

MY LINEAR REGRESSION::


,Metric,Train,Test
0,MAE,713.56,708.55
1,RMSE,1037.76,1025.99
2,R²,0.5785,0.5859


### v. Compare with sklearn LinearRegression

In [239]:
from sklearn.linear_model import LinearRegression

sk_model = LinearRegression()
sk_model.fit(X_train_scaled, y_train)

y_train_pred_sk = sk_model.predict(X_train_scaled)
y_test_pred_sk = sk_model.predict(X_test_scaled)

mae_train_sk = np.mean(np.abs(y_train - y_train_pred_sk))
mae_test_sk = np.mean(np.abs(y_test - y_test_pred_sk))

rmse_train_sk = np.sqrt(np.mean((y_train - y_train_pred_sk) ** 2))
rmse_test_sk = np.sqrt(np.mean((y_test - y_test_pred_sk) ** 2))

r2_train_sk = my_r2_score(y_train, y_train_pred_sk)
r2_test_sk = my_r2_score(y_test, y_test_pred_sk)


results_linear_full = pd.DataFrame({
    'Model': ['my_LinearRegression', 'sklearn_LinearRegression', 'Difference'],
    'MAE_train': [f"{mae_train_my:.2f}", f"{mae_train_sk:.2f}", "-"],
    'MAE_test': [f"{mae_test_my:.2f}", f"{mae_test_sk:.2f}", f"{abs(mae_test_my - mae_test_sk):.8f}"],
    'RMSE_train': [f"{rmse_train_my:.2f}", f"{rmse_train_sk:.2f}", "-"],
    'RMSE_test': [f"{rmse_test_my:.2f}", f"{rmse_test_sk:.2f}", f"{abs(rmse_test_my - rmse_test_sk):.8f}"],
    'R2_train': [f"{r2_train_my:.4f}", f"{r2_train_sk:.4f}", "-"],
    'R2_test': [f"{r2_test_my:.4f}", f"{r2_test_sk:.4f}", f"{abs(r2_test_my - r2_test_sk):.8f}"]
})

display(results_linear_full)

,Model,MAE_train,MAE_test,RMSE_train,RMSE_test,R2_train,R2_test
0,my_LinearRegression,713.56,708.55,1037.76,1025.99,0.5785,0.5859
1,sklearn_LinearRegression,713.56,708.55,1037.76,1025.99,0.5785,0.5859
2,Difference,-,0.00000000,-,0.00000000,-,0.00000000


## 5. Regularized models implementation — Ridge, Lasso, ElasticNet

### i. Implement Ridge, Lasso, ElasticNet algorithms

#### a. Implement Ridge Regression (L2 regularization)

In [240]:
class MyRidgeRegression:

    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.weights = None
        
    def fit(self, X, y):
        X_with_bias = np.c_[np.ones(X.shape[0]), X]
        n_features = X_with_bias.shape[1]
        # Регуляризационная матрица (bias не регуляризуем)
        reg_matrix = np.eye(n_features)
        reg_matrix[0, 0] = 0
        XTX = X_with_bias.T @ X_with_bias
        self.weights = np.linalg.pinv(XTX + self.alpha * reg_matrix) @ X_with_bias.T @ y
        return self
    
    def predict(self, X):
        X_with_bias = np.c_[np.ones(X.shape[0]), X]
        return X_with_bias @ self.weights

print("MyRidgeRegression создан")

MyRidgeRegression создан


#### b. Implement Lasso Regression (L1 regularization)

In [241]:
rng = np.random.default_rng(seed=21)

class MyLassoRegression:

    def __init__(self, lambda_=0.01, max_iter=100000, step=0.001):
        self.lambda_ = lambda_
        self.max_iter = max_iter
        self.step = step
        self.weights = None
        
    def fit(self, X, y):
        X_with_bias = np.c_[np.ones(X.shape[0]), X]
        n_samples, n_features = X_with_bias.shape
        self.weights = np.zeros(n_features)
        
        for _ in range(self.max_iter):
            idx = rng.integers(0, n_samples - 1)
            x_i = X_with_bias[idx]
            y_i = y[idx]
            
            y_pred = np.dot(x_i, self.weights)
            error = y_i - y_pred
            
            grad = -2 * error * x_i + self.lambda_ * np.sign(self.weights)
            self.weights -= self.step * grad
            
        return self
    
    def predict(self, X):
        X_with_bias = np.c_[np.ones(X.shape[0]), X]
        return X_with_bias @ self.weights

print("MyLassoRegression создан")

MyLassoRegression создан


#### b. Implement ElasticNet Regression (L1 + L2 regularization)

In [242]:
class MyElasticNetRegression:

    def __init__(self, l1_ratio=0.5, alpha=0.01, max_iter=100000, step=0.001):
        self.l1_coef = alpha * l1_ratio
        self.l2_coef = alpha * (1 - l1_ratio)
        self.max_iter = max_iter
        self.step = step
        self.weights = None
        
    def fit(self, X, y):
        X_with_bias = np.c_[np.ones(X.shape[0]), X]
        n_samples, n_features = X_with_bias.shape
        self.weights = np.zeros(n_features)
        
        for _ in range(self.max_iter):
            idx = rng.integers(0, n_samples - 1)
            x_i = X_with_bias[idx]
            y_i = y[idx]
            
            y_pred = np.dot(x_i, self.weights)
            error = y_i - y_pred
            
            grad = -2 * error * x_i
            grad += self.l1_coef * np.sign(self.weights)
            grad += 2 * self.l2_coef * self.weights
            
            self.weights -= self.step * grad
            
        return self
    
    def predict(self, X):
        X_with_bias = np.c_[np.ones(X.shape[0]), X]
        return X_with_bias @ self.weights

print("MyElasticNetRegression создан")

MyElasticNetRegression создан


### ii. Make predictions with your algorithm and estimate the model with MAE, RMSE and R2 metrics.

#### a. My Ridge Regression

In [243]:
my_ridge = MyRidgeRegression(alpha=1.0)
my_ridge.fit(X_train_scaled, y_train)

y_train_pred_ridge = my_ridge.predict(X_train_scaled)
y_test_pred_ridge = my_ridge.predict(X_test_scaled)

mae_train_ridge = np.mean(np.abs(y_train - y_train_pred_ridge))
mae_test_ridge = np.mean(np.abs(y_test - y_test_pred_ridge))

rmse_train_ridge = np.sqrt(np.mean((y_train - y_train_pred_ridge) ** 2))
rmse_test_ridge = np.sqrt(np.mean((y_test - y_test_pred_ridge) ** 2))

r2_train_ridge = my_r2_score(y_train, y_train_pred_ridge)
r2_test_ridge = my_r2_score(y_test, y_test_pred_ridge)

print_results_table(
    "MY RIDGE REGRESSION",
    mae_train_ridge, mae_test_ridge,
    rmse_train_ridge, rmse_test_ridge,
    r2_train_ridge, r2_test_ridge
)

MY RIDGE REGRESSION:


,Metric,Train,Test
0,MAE,713.55,708.55
1,RMSE,1037.76,1025.99
2,R²,0.5785,0.5859


#### b. Lasso Regression

In [244]:
print(f"Тип X_train_scaled: {type(X_train_scaled)}")
print(f"Тип y_train: {type(y_train)}")
print(f"X_train_scaled shape: {X_train_scaled.shape}")

Тип X_train_scaled: <class 'numpy.ndarray'>
Тип y_train: <class 'pandas.Series'>
X_train_scaled shape: (38703, 22)


In [245]:
y_train_np = y_train.values
y_test_np = y_test.values

my_lasso = MyLassoRegression(lambda_=0.01, max_iter=100000, step=0.001)
my_lasso.fit(X_train_scaled, y_train_np)

y_train_pred_lasso = my_lasso.predict(X_train_scaled)
y_test_pred_lasso = my_lasso.predict(X_test_scaled)

mae_train_lasso = np.mean(np.abs(y_train_np - y_train_pred_lasso))
mae_test_lasso = np.mean(np.abs(y_test_np - y_test_pred_lasso))

rmse_train_lasso = np.sqrt(np.mean((y_train_np - y_train_pred_lasso) ** 2))
rmse_test_lasso = np.sqrt(np.mean((y_test_np - y_test_pred_lasso) ** 2))

r2_train_lasso = my_r2_score(y_train_np, y_train_pred_lasso)
r2_test_lasso = my_r2_score(y_test_np, y_test_pred_lasso)

print_results_table(
    "MY LASSO REGRESSION",
    mae_train_lasso, mae_test_lasso,
    rmse_train_lasso, rmse_test_lasso,
    r2_train_lasso, r2_test_lasso
)

MY LASSO REGRESSION:


,Metric,Train,Test
0,MAE,738.51,734.00
1,RMSE,1054.91,1048.19
2,R²,0.5645,0.5678


#### c. ElasticNet Regression

In [246]:
y_train_np = y_train.values if isinstance(y_train, pd.Series) else y_train
y_test_np = y_test.values if isinstance(y_test, pd.Series) else y_test

my_elastic = MyElasticNetRegression(l1_ratio=0.5, alpha=0.01, max_iter=100000, step=0.001)
my_elastic.fit(X_train_scaled, y_train_np)

y_train_pred_elastic = my_elastic.predict(X_train_scaled)
y_test_pred_elastic = my_elastic.predict(X_test_scaled)

mae_train_elastic = np.mean(np.abs(y_train_np - y_train_pred_elastic))
mae_test_elastic = np.mean(np.abs(y_test_np - y_test_pred_elastic))

rmse_train_elastic = np.sqrt(np.mean((y_train_np - y_train_pred_elastic) ** 2))
rmse_test_elastic = np.sqrt(np.mean((y_test_np - y_test_pred_elastic) ** 2))

r2_train_elastic = my_r2_score(y_train_np, y_train_pred_elastic)
r2_test_elastic = my_r2_score(y_test_np, y_test_pred_elastic)

print_results_table(
    "MY ELASTICNET REGRESSION",
    mae_train_elastic, mae_test_elastic,
    rmse_train_elastic, rmse_test_elastic,
    r2_train_elastic, r2_test_elastic
)

MY ELASTICNET REGRESSION:


,Metric,Train,Test
0,MAE,718.85,715.56
1,RMSE,1054.70,1042.95
2,R²,0.5646,0.5721


### iii. Initialize Ridge(), Lasso(), and ElasticNet() from sklearn.linear_model, fit the model, and make predictions for the training and test samples as in the previous lesson.

#### a. Sklearn Ridge Regression

In [247]:
from sklearn.linear_model import Ridge

sk_ridge = Ridge(alpha=1.0)
sk_ridge.fit(X_train_scaled, y_train)

y_train_pred_sk_ridge = sk_ridge.predict(X_train_scaled)
y_test_pred_sk_ridge = sk_ridge.predict(X_test_scaled)

mae_train_sk_ridge = np.mean(np.abs(y_train - y_train_pred_sk_ridge))
mae_test_sk_ridge = np.mean(np.abs(y_test - y_test_pred_sk_ridge))
rmse_train_sk_ridge = np.sqrt(np.mean((y_train - y_train_pred_sk_ridge) ** 2))
rmse_test_sk_ridge = np.sqrt(np.mean((y_test - y_test_pred_sk_ridge) ** 2))
r2_train_sk_ridge = my_r2_score(y_train, y_train_pred_sk_ridge)
r2_test_sk_ridge = my_r2_score(y_test, y_test_pred_sk_ridge)

print_results_table(
    "SKLEARN RIDGE REGRESSION",
    mae_train_sk_ridge, mae_test_sk_ridge,
    rmse_train_sk_ridge, rmse_test_sk_ridge,
    r2_train_sk_ridge, r2_test_sk_ridge
)

SKLEARN RIDGE REGRESSION:


,Metric,Train,Test
0,MAE,713.55,708.55
1,RMSE,1037.76,1025.99
2,R²,0.5785,0.5859


#### b. Sklearn Lasso Regression

In [248]:
from sklearn.linear_model import Lasso

sk_lasso = Lasso(alpha=0.01, max_iter=100000)
sk_lasso.fit(X_train_scaled, y_train)

y_train_pred_sk_lasso = sk_lasso.predict(X_train_scaled)
y_test_pred_sk_lasso = sk_lasso.predict(X_test_scaled)

mae_train_sk_lasso = np.mean(np.abs(y_train - y_train_pred_sk_lasso))
mae_test_sk_lasso = np.mean(np.abs(y_test - y_test_pred_sk_lasso))
rmse_train_sk_lasso = np.sqrt(np.mean((y_train - y_train_pred_sk_lasso) ** 2))
rmse_test_sk_lasso = np.sqrt(np.mean((y_test - y_test_pred_sk_lasso) ** 2))
r2_train_sk_lasso = my_r2_score(y_train, y_train_pred_sk_lasso)
r2_test_sk_lasso = my_r2_score(y_test, y_test_pred_sk_lasso)

print_results_table(
    "SKLEARN LASSO REGRESSION",
    mae_train_sk_lasso, mae_test_sk_lasso,
    rmse_train_sk_lasso, rmse_test_sk_lasso,
    r2_train_sk_lasso, r2_test_sk_lasso
)

SKLEARN LASSO REGRESSION:


,Metric,Train,Test
0,MAE,713.55,708.55
1,RMSE,1037.76,1025.99
2,R²,0.5785,0.5859


#### c. Sklearn ElasticNet Regression

In [249]:
from sklearn.linear_model import ElasticNet

sk_elastic = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=100000)
sk_elastic.fit(X_train_scaled, y_train)

y_train_pred_sk_elastic = sk_elastic.predict(X_train_scaled)
y_test_pred_sk_elastic = sk_elastic.predict(X_test_scaled)

mae_train_sk_elastic = np.mean(np.abs(y_train - y_train_pred_sk_elastic))
mae_test_sk_elastic = np.mean(np.abs(y_test - y_test_pred_sk_elastic))
rmse_train_sk_elastic = np.sqrt(np.mean((y_train - y_train_pred_sk_elastic) ** 2))
rmse_test_sk_elastic = np.sqrt(np.mean((y_test - y_test_pred_sk_elastic) ** 2))
r2_train_sk_elastic = my_r2_score(y_train, y_train_pred_sk_elastic)
r2_test_sk_elastic = my_r2_score(y_test, y_test_pred_sk_elastic)

print_results_table(
    "SKLEARN ELASTICNET REGRESSION",
    mae_train_sk_elastic, mae_test_sk_elastic,
    rmse_train_sk_elastic, rmse_test_sk_elastic,
    r2_train_sk_elastic, r2_test_sk_elastic
)

SKLEARN ELASTICNET REGRESSION:


,Metric,Train,Test
0,MAE,713.43,708.30
1,RMSE,1037.78,1025.90
2,R²,0.5785,0.5860


### v. Store the metrics as in the previous lesson in a table with columns model, train, test for MAE table, RMSE table, and R2 coefficient.

In [250]:
models_data = [
    ("my_LinearRegression", mae_train_my, mae_test_my, rmse_train_my, rmse_test_my, r2_train_my, r2_test_my),
    ("sklearn_LinearRegression", mae_train_sk, mae_test_sk, rmse_train_sk, rmse_test_sk, r2_train_sk, r2_test_sk),
    ("my_Ridge", mae_train_ridge, mae_test_ridge, rmse_train_ridge, rmse_test_ridge, r2_train_ridge, r2_test_ridge),
    ("my_Lasso", mae_train_lasso, mae_test_lasso, rmse_train_lasso, rmse_test_lasso, r2_train_lasso, r2_test_lasso),
    ("my_ElasticNet", mae_train_elastic, mae_test_elastic, rmse_train_elastic, rmse_test_elastic, r2_train_elastic, r2_test_elastic),
    ("sklearn_Ridge", mae_train_sk_ridge, mae_test_sk_ridge, rmse_train_sk_ridge, rmse_test_sk_ridge, r2_train_sk_ridge, r2_test_sk_ridge),
    ("sklearn_Lasso", mae_train_sk_lasso, mae_test_sk_lasso, rmse_train_sk_lasso, rmse_test_sk_lasso, r2_train_sk_lasso, r2_test_sk_lasso),
    ("sklearn_ElasticNet", mae_train_sk_elastic, mae_test_sk_elastic, rmse_train_sk_elastic, rmse_test_sk_elastic, r2_train_sk_elastic, r2_test_sk_elastic)
]

results_regularized = pd.DataFrame(
    np.nan,
    index=pd.MultiIndex.from_product(
        [[name for name, _, _, _, _, _, _ in models_data], ["train", "test"]],
        names=["model", "split"],
    ),
    columns=["MAE", "RMSE", "R2"],
)

for name, mae_tr, mae_te, rmse_tr, rmse_te, r2_tr, r2_te in models_data:
    results_regularized.loc[(name, "train"), "MAE"] = mae_tr
    results_regularized.loc[(name, "train"), "RMSE"] = rmse_tr
    results_regularized.loc[(name, "train"), "R2"] = r2_tr
    results_regularized.loc[(name, "test"), "MAE"] = mae_te
    results_regularized.loc[(name, "test"), "RMSE"] = rmse_te
    results_regularized.loc[(name, "test"), "R2"] = r2_te

for col in ["MAE", "RMSE"]:
    results_regularized[col] = results_regularized[col].map(lambda x: f"{x:.2f}")
results_regularized["R2"] = results_regularized["R2"].map(lambda x: f"{x:.4f}")

display(results_regularized)

MAE     RMSE      R2
model                    split                         
my_LinearRegression      train  713.56  1037.76  0.5785
                         test   708.55  1025.99  0.5859
sklearn_LinearRegression train  713.56  1037.76  0.5785
                         test   708.55  1025.99  0.5859
my_Ridge                 train  713.55  1037.76  0.5785
                         test   708.55  1025.99  0.5859
my_Lasso                 train  738.51  1054.91  0.5645
                         test   734.00  1048.19  0.5678
my_ElasticNet            train  718.85  1054.70  0.5646
                         test   715.56  1042.95  0.5721
sklearn_Ridge            train  713.55  1037.76  0.5785
                         test   708.55  1025.99  0.5859
sklearn_Lasso            train  713.55  1037.76  0.5785
                         test   708.55  1025.99  0.5859
sklearn_ElasticNet       train  713.43  1037.78  0.5785
                         test   708.30  1025.90  0.5860

## 6. Feature normalization

### i. Examples of why and where feature normalization is mandatory

**Когда нормализация обязательна:**
- признаки в разных масштабах (возраст 0–100 и доход 0–100 000) — без нормализации градиентный спуск работает плохо.
- обязательна для стабильности градиентов.
- **Методы, основанные на расстояниях (KNN, SVM, K-Means):** иначе признаки с большими значениями доминируют.
- 
**Когда нормализация не обязательна (или вредна):**
- **Деревья (XGBoost, Random Forest):** инвариантны к масштабированию.
- **Линейные модели с интерпретируемыми коэффициентами:** если признаки уже в одинаковых единицах (например, проценты).
- **Эмбеддинги:** нормализация входов перед эмбеддингами избыточна.

### ii. Write a mathematical formula for this method (MinMaxScaler)

$$
X_{\textrm{scaled}} = \frac{X - x_{min}}{x_{max} - x_{min}}
$$

### iii. Implement your own function or class for MinMaxScaler feature normalization

In [251]:
class MyMinMaxScaler:
    """
    MinMaxScaler: scales features to a given range, typically [0, 1]
    Formula: x_scaled = (x - x_min) / (x_max - x_min)
    """
    
    def __init__(self, feature_range=(0, 1)):
        self.feature_range = feature_range
        self.min_range = feature_range[0]
        self.max_range = feature_range[1]
        self.min_ = None
        self.scale_ = None  # scale factor = max_range - min_range
        self.data_min_ = None
        self.data_max_ = None
        
    def fit(self, X):
        """
        Compute min and max for each feature
        X: numpy array (n_samples, n_features)
        """
        self.data_min_ = np.min(X, axis=0)
        self.data_max_ = np.max(X, axis=0)
        
        data_range = self.data_max_ - self.data_min_
        data_range[data_range == 0] = 1
        
        self.scale_ = (self.max_range - self.min_range) / data_range
        return self
    
    def transform(self, X):
        """
        Scale features to the given range
        X: numpy array (n_samples, n_features)
        """
        if self.data_min_ is None:
            raise ValueError("Scaler not fitted. Call fit() first.")
        
        X_scaled = (X - self.data_min_) * self.scale_ + self.min_range
        return X_scaled
    
    def fit_transform(self, X):
        """
        Fit and transform in one step
        """
        self.fit(X)
        return self.transform(X)

print("MyMinMaxScaler created")

MyMinMaxScaler created


### iv. Initialize MinMaxScaler() from sklearn.preprocessing.

In [252]:
from sklearn.preprocessing import MinMaxScaler

sklearn_minmax = MinMaxScaler(feature_range=(0, 1))
sklearn_minmax.fit(X_train)

X_train_sk_minmax = sklearn_minmax.transform(X_train)
X_test_sk_minmax = sklearn_minmax.transform(X_test)

print("Sklearn MinMaxScaler initialized and fitted")
print(f"X_train shape after transform: {X_train_sk_minmax.shape}")
print(f"X_test shape after transform: {X_test_sk_minmax.shape}")

Sklearn MinMaxScaler initialized and fitted
X_train shape after transform: (38703, 22)
X_test shape after transform: (9676, 22)


### v. Compare the feature normalization with your own method and with sklearn.

In [253]:
import pandas as pd

test_data = np.array([[-1, 2], [-0.5, 6], [0, 10], [1, 18]])
my_scaler = MyMinMaxScaler(feature_range=(0, 1))
my_result = my_scaler.fit_transform(test_data)

sk_scaler = MinMaxScaler(feature_range=(0, 1))
sk_result = sk_scaler.fit_transform(test_data)

rows = []
for col_idx in range(test_data.shape[1]):
    for row_idx in range(test_data.shape[0]):
        rows.append({
            'Feature': f'Col{col_idx}',
            'Row': row_idx,
            'Original': test_data[row_idx, col_idx],
            'MyMinMax': my_result[row_idx, col_idx],
            'SklearnMinMax': sk_result[row_idx, col_idx]
        })

df_comparison = pd.DataFrame(rows)
df_comparison = df_comparison.set_index(['Feature', 'Row'])


print("MinMaxScaler Comparison:")
display(df_comparison)

MinMaxScaler Comparison:


Original  MyMinMax  SklearnMinMax
Feature Row                                   
Col0    0        -1.0      0.00           0.00
        1        -0.5      0.25           0.25
        2         0.0      0.50           0.50
        3         1.0      1.00           1.00
Col1    0         2.0      0.00           0.00
        1         6.0      0.25           0.25
        2        10.0      0.50           0.50
        3        18.0      1.00           1.00

### vi. Repeat the steps from b to e for another normalization method StandardScaler.

#### a. Implement your own function or class for StandardScaler feature normalization

In [254]:
class MyStandardScaler:
    
    def __init__(self):
        self.mean_ = None
        self.scale_ = None  # standard deviation
        self.var_ = None    # variance
        
    def fit(self, X):
        """
        Compute mean and standard deviation for each feature
        X: numpy array (n_samples, n_features)
        """
        self.mean_ = np.mean(X, axis=0)
        self.var_ = np.var(X, axis=0, ddof=0)  # ddof=0 for population variance (like sklearn)
        self.scale_ = np.sqrt(self.var_)
        
        self.scale_[self.scale_ == 0] = 1
        return self
    
    def transform(self, X):
        if self.mean_ is None:
            raise ValueError("Scaler not fitted. Call fit() first.")
        
        X_scaled = (X - self.mean_) / self.scale_
        return X_scaled
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

print("MyStandardScaler created")

MyStandardScaler created


#### b. Initialize StandardScaler from sklearn.preprocessing

In [255]:
from sklearn.preprocessing import StandardScaler

sklearn_standard = StandardScaler()
sklearn_standard.fit(X_train)

X_train_sk_standard = sklearn_standard.transform(X_train)
X_test_sk_standard = sklearn_standard.transform(X_test)

print("Sklearn StandardScaler initialized and fitted")
print(f"X_train shape after transform: {X_train_sk_standard.shape}")
print(f"X_test shape after transform: {X_test_sk_standard.shape}")

Sklearn StandardScaler initialized and fitted
X_train shape after transform: (38703, 22)
X_test shape after transform: (9676, 22)


#### c. Compare MyStandardScaler with sklearn StandardScaler

In [256]:
from sklearn.preprocessing import StandardScaler

test_data = np.array([[-1, 2], [-0.5, 6], [0, 10], [1, 18]])

my_scaler_std = MyStandardScaler()
my_result_std = my_scaler_std.fit_transform(test_data)

sk_scaler_std = StandardScaler()
sk_result_std = sk_scaler_std.fit_transform(test_data)

rows_std = []
for col_idx in range(test_data.shape[1]):
    for row_idx in range(test_data.shape[0]):
        rows_std.append({
            'Feature': f'Col{col_idx}',
            'Row': row_idx,
            'Original': test_data[row_idx, col_idx],
            'MyStandard': my_result_std[row_idx, col_idx],
            'SklearnStandard': sk_result_std[row_idx, col_idx]
        })

df_standard_comparison = pd.DataFrame(rows_std)
df_standard_comparison = df_standard_comparison.set_index(['Feature', 'Row'])

print("StandardScaler Comparison")
display(df_standard_comparison)

StandardScaler Comparison


Original  MyStandard  SklearnStandard
Feature Row                                       
Col0    0        -1.0   -1.183216        -1.183216
        1        -0.5   -0.507093        -0.507093
        2         0.0    0.169031         0.169031
        3         1.0    1.521278         1.521278
Col1    0         2.0   -1.183216        -1.183216
        1         6.0   -0.507093        -0.507093
        2        10.0    0.169031         0.169031
        3        18.0    1.521278         1.521278

#### d. MinMaxScaler and StandardScaler Results

In [257]:
test_data = np.array([[-1, 2], [-0.5, 6], [0, 10], [1, 18]])

my_minmax = MyMinMaxScaler(feature_range=(0, 1))
my_minmax_result = my_minmax.fit_transform(test_data)

sk_minmax = MinMaxScaler(feature_range=(0, 1))
sk_minmax_result = sk_minmax.fit_transform(test_data)

my_std = MyStandardScaler()
my_std_result = my_std.fit_transform(test_data)

sk_std = StandardScaler()
sk_std_result = sk_std.fit_transform(test_data)

rows = []
for col_idx in range(test_data.shape[1]):
    for row_idx in range(test_data.shape[0]):
        rows.append({
            'Feature': f'Col{col_idx}',
            'Row': row_idx,
            'Original': test_data[row_idx, col_idx],
            'MyMinMax': my_minmax_result[row_idx, col_idx],
            'SklearnMinMax': sk_minmax_result[row_idx, col_idx],
            'MyStandard': my_std_result[row_idx, col_idx],
            'SklearnStandard': sk_std_result[row_idx, col_idx]
        })

df_combined = pd.DataFrame(rows)
df_combined = df_combined.set_index(['Feature', 'Row'])

display(df_combined)

Original  MyMinMax  SklearnMinMax  MyStandard  SklearnStandard
Feature Row                                                                
Col0    0        -1.0      0.00           0.00   -1.183216        -1.183216
        1        -0.5      0.25           0.25   -0.507093        -0.507093
        2         0.0      0.50           0.50    0.169031         0.169031
        3         1.0      1.00           1.00    1.521278         1.521278
Col1    0         2.0      0.00           0.00   -1.183216        -1.183216
        1         6.0      0.25           0.25   -0.507093        -0.507093
        2        10.0      0.50           0.50    0.169031         0.169031
        3        18.0      1.00           1.00    1.521278         1.521278

## 7. Fit custom and sklearn models with normalized data

### i. Fit all models — Linear Regression, Ridge, Lasso, and ElasticNet — with MinMaxScaler.

#### a. Apply MinMaxScaler to data

In [258]:
minmax_scaler = MinMaxScaler()
X_train_minmax = minmax_scaler.fit_transform(X_train)
X_test_minmax = minmax_scaler.transform(X_test)

print("MinMaxScaler applied")
print(f"X_train_minmax shape: {X_train_minmax.shape}")
print(f"X_test_minmax shape: {X_test_minmax.shape}")

MinMaxScaler applied
X_train_minmax shape: (38703, 22)
X_test_minmax shape: (9676, 22)


#### b. Linear Regression with MinMaxScaler

In [259]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

lr_minmax = LinearRegression()
lr_minmax.fit(X_train_minmax, y_train)

y_train_pred_lr_mm = lr_minmax.predict(X_train_minmax)
y_test_pred_lr_mm = lr_minmax.predict(X_test_minmax)

mae_train_lr_mm = np.mean(np.abs(y_train - y_train_pred_lr_mm))
mae_test_lr_mm = np.mean(np.abs(y_test - y_test_pred_lr_mm))
rmse_train_lr_mm = np.sqrt(np.mean((y_train - y_train_pred_lr_mm) ** 2))
rmse_test_lr_mm = np.sqrt(np.mean((y_test - y_test_pred_lr_mm) ** 2))
r2_train_lr_mm = my_r2_score(y_train, y_train_pred_lr_mm)
r2_test_lr_mm = my_r2_score(y_test, y_test_pred_lr_mm)

print_results_table(
    "LINEAR REGRESSION (MinMaxScaler)",
    mae_train_lr_mm, mae_test_lr_mm,
    rmse_train_lr_mm, rmse_test_lr_mm,
    r2_train_lr_mm, r2_test_lr_mm
)

LINEAR REGRESSION (MinMaxScaler):


,Metric,Train,Test
0,MAE,713.56,708.55
1,RMSE,1037.76,1025.99
2,R²,0.5785,0.5859


#### c. Ridge Regression with MinMaxScaler

In [260]:
ridge_minmax = Ridge(alpha=1.0)
ridge_minmax.fit(X_train_minmax, y_train)

y_train_pred_ridge_mm = ridge_minmax.predict(X_train_minmax)
y_test_pred_ridge_mm = ridge_minmax.predict(X_test_minmax)

mae_train_ridge_mm = np.mean(np.abs(y_train - y_train_pred_ridge_mm))
mae_test_ridge_mm = np.mean(np.abs(y_test - y_test_pred_ridge_mm))
rmse_train_ridge_mm = np.sqrt(np.mean((y_train - y_train_pred_ridge_mm) ** 2))
rmse_test_ridge_mm = np.sqrt(np.mean((y_test - y_test_pred_ridge_mm) ** 2))
r2_train_ridge_mm = my_r2_score(y_train, y_train_pred_ridge_mm)
r2_test_ridge_mm = my_r2_score(y_test, y_test_pred_ridge_mm)

print_results_table(
    "RIDGE REGRESSION (MinMaxScaler)",
    mae_train_ridge_mm, mae_test_ridge_mm,
    rmse_train_ridge_mm, rmse_test_ridge_mm,
    r2_train_ridge_mm, r2_test_ridge_mm
)

RIDGE REGRESSION (MinMaxScaler):


,Metric,Train,Test
0,MAE,713.71,708.51
1,RMSE,1037.82,1026.06
2,R²,0.5785,0.5858


#### c. Lasso Regression with MinMaxScaler

In [261]:
lasso_minmax = Lasso(alpha=0.01, max_iter=100000)
lasso_minmax.fit(X_train_minmax, y_train)

y_train_pred_lasso_mm = lasso_minmax.predict(X_train_minmax)
y_test_pred_lasso_mm = lasso_minmax.predict(X_test_minmax)

mae_train_lasso_mm = np.mean(np.abs(y_train - y_train_pred_lasso_mm))
mae_test_lasso_mm = np.mean(np.abs(y_test - y_test_pred_lasso_mm))
rmse_train_lasso_mm = np.sqrt(np.mean((y_train - y_train_pred_lasso_mm) ** 2))
rmse_test_lasso_mm = np.sqrt(np.mean((y_test - y_test_pred_lasso_mm) ** 2))
r2_train_lasso_mm = my_r2_score(y_train, y_train_pred_lasso_mm)
r2_test_lasso_mm = my_r2_score(y_test, y_test_pred_lasso_mm)

print_results_table(
    "LASSO REGRESSION (MinMaxScaler)",
    mae_train_lasso_mm, mae_test_lasso_mm,
    rmse_train_lasso_mm, rmse_test_lasso_mm,
    r2_train_lasso_mm, r2_test_lasso_mm
)

LASSO REGRESSION (MinMaxScaler):


,Metric,Train,Test
0,MAE,713.55,708.54
1,RMSE,1037.76,1025.99
2,R²,0.5785,0.5859


#### d. ElasticNet Regression with MinMaxScaler

In [262]:
elastic_minmax = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=100000)
elastic_minmax.fit(X_train_minmax, y_train)

y_train_pred_elastic_mm = elastic_minmax.predict(X_train_minmax)
y_test_pred_elastic_mm = elastic_minmax.predict(X_test_minmax)

mae_train_elastic_mm = np.mean(np.abs(y_train - y_train_pred_elastic_mm))
mae_test_elastic_mm = np.mean(np.abs(y_test - y_test_pred_elastic_mm))
rmse_train_elastic_mm = np.sqrt(np.mean((y_train - y_train_pred_elastic_mm) ** 2))
rmse_test_elastic_mm = np.sqrt(np.mean((y_test - y_test_pred_elastic_mm) ** 2))
r2_train_elastic_mm = my_r2_score(y_train, y_train_pred_elastic_mm)
r2_test_elastic_mm = my_r2_score(y_test, y_test_pred_elastic_mm)

print_results_table(
    "ELASTICNET REGRESSION (MinMaxScaler)",
    mae_train_elastic_mm, mae_test_elastic_mm,
    rmse_train_elastic_mm, rmse_test_elastic_mm,
    r2_train_elastic_mm, r2_test_elastic_mm
)

ELASTICNET REGRESSION (MinMaxScaler):


,Metric,Train,Test
0,MAE,774.60,766.73
1,RMSE,1139.10,1132.46
2,R²,0.4922,0.4955


#### 6. All Models with MinMaxScaler

In [263]:
models_minmax = [
    ("LinearRegression", mae_train_lr_mm, mae_test_lr_mm, rmse_train_lr_mm, rmse_test_lr_mm, r2_train_lr_mm, r2_test_lr_mm),
    ("Ridge", mae_train_ridge_mm, mae_test_ridge_mm, rmse_train_ridge_mm, rmse_test_ridge_mm, r2_train_ridge_mm, r2_test_ridge_mm),
    ("Lasso", mae_train_lasso_mm, mae_test_lasso_mm, rmse_train_lasso_mm, rmse_test_lasso_mm, r2_train_lasso_mm, r2_test_lasso_mm),
    ("ElasticNet", mae_train_elastic_mm, mae_test_elastic_mm, rmse_train_elastic_mm, rmse_test_elastic_mm, r2_train_elastic_mm, r2_test_elastic_mm)
]

create_summary_table(models_minmax, "SUMMARY - MODELS WITH MINMAX SCALER")

SUMMARY - MODELS WITH MINMAX SCALER


MAE     RMSE      R2
model            split                         
LinearRegression train  713.56  1037.76  0.5785
                 test   708.55  1025.99  0.5859
Ridge            train  713.71  1037.82  0.5785
                 test   708.51  1026.06  0.5858
Lasso            train  713.55  1037.76  0.5785
                 test   708.54  1025.99  0.5859
ElasticNet       train  774.60  1139.10  0.4922
                 test   766.73  1132.46  0.4955

### ii. Fit all models — Linear Regression, Ridge, Lasso, and ElasticNet — with StandardScaler.

#### a. Apply StandardScaler to data

In [264]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()
X_train_std = standard_scaler.fit_transform(X_train)
X_test_std = standard_scaler.transform(X_test)

print("StandardScaler applied")
print(f"X_train_std shape: {X_train_std.shape}")
print(f"X_test_std shape: {X_test_std.shape}")

StandardScaler applied
X_train_std shape: (38703, 22)
X_test_std shape: (9676, 22)


#### b. Linear Regression with StandardScaler

In [265]:
from sklearn.linear_model import LinearRegression

lr_std = LinearRegression()
lr_std.fit(X_train_std, y_train)

y_train_pred_lr_std = lr_std.predict(X_train_std)
y_test_pred_lr_std = lr_std.predict(X_test_std)

mae_train_lr_std = np.mean(np.abs(y_train - y_train_pred_lr_std))
mae_test_lr_std = np.mean(np.abs(y_test - y_test_pred_lr_std))
rmse_train_lr_std = np.sqrt(np.mean((y_train - y_train_pred_lr_std) ** 2))
rmse_test_lr_std = np.sqrt(np.mean((y_test - y_test_pred_lr_std) ** 2))
r2_train_lr_std = my_r2_score(y_train, y_train_pred_lr_std)
r2_test_lr_std = my_r2_score(y_test, y_test_pred_lr_std)

print_results_table(
    "LINEAR REGRESSION (StandardScaler)",
    mae_train_lr_std, mae_test_lr_std,
    rmse_train_lr_std, rmse_test_lr_std,
    r2_train_lr_std, r2_test_lr_std
)

LINEAR REGRESSION (StandardScaler):


,Metric,Train,Test
0,MAE,713.56,708.55
1,RMSE,1037.76,1025.99
2,R²,0.5785,0.5859


#### c. Ridge Regression with StandardScaler

In [266]:
from sklearn.linear_model import Ridge

ridge_std = Ridge(alpha=1.0)
ridge_std.fit(X_train_std, y_train)

y_train_pred_ridge_std = ridge_std.predict(X_train_std)
y_test_pred_ridge_std = ridge_std.predict(X_test_std)

mae_train_ridge_std = np.mean(np.abs(y_train - y_train_pred_ridge_std))
mae_test_ridge_std = np.mean(np.abs(y_test - y_test_pred_ridge_std))
rmse_train_ridge_std = np.sqrt(np.mean((y_train - y_train_pred_ridge_std) ** 2))
rmse_test_ridge_std = np.sqrt(np.mean((y_test - y_test_pred_ridge_std) ** 2))
r2_train_ridge_std = my_r2_score(y_train, y_train_pred_ridge_std)
r2_test_ridge_std = my_r2_score(y_test, y_test_pred_ridge_std)

print_results_table(
    "RIDGE REGRESSION (StandardScaler)",
    mae_train_ridge_std, mae_test_ridge_std,
    rmse_train_ridge_std, rmse_test_ridge_std,
    r2_train_ridge_std, r2_test_ridge_std
)

RIDGE REGRESSION (StandardScaler):


,Metric,Train,Test
0,MAE,713.55,708.55
1,RMSE,1037.76,1025.99
2,R²,0.5785,0.5859


#### c. Lasso Regression with StandardScaler

In [267]:
from sklearn.linear_model import Lasso

lasso_std = Lasso(alpha=0.01, max_iter=100000)
lasso_std.fit(X_train_std, y_train)

y_train_pred_lasso_std = lasso_std.predict(X_train_std)
y_test_pred_lasso_std = lasso_std.predict(X_test_std)

mae_train_lasso_std = np.mean(np.abs(y_train - y_train_pred_lasso_std))
mae_test_lasso_std = np.mean(np.abs(y_test - y_test_pred_lasso_std))
rmse_train_lasso_std = np.sqrt(np.mean((y_train - y_train_pred_lasso_std) ** 2))
rmse_test_lasso_std = np.sqrt(np.mean((y_test - y_test_pred_lasso_std) ** 2))
r2_train_lasso_std = my_r2_score(y_train, y_train_pred_lasso_std)
r2_test_lasso_std = my_r2_score(y_test, y_test_pred_lasso_std)

print_results_table(
    "LASSO REGRESSION (StandardScaler)",
    mae_train_lasso_std, mae_test_lasso_std,
    rmse_train_lasso_std, rmse_test_lasso_std,
    r2_train_lasso_std, r2_test_lasso_std
)

LASSO REGRESSION (StandardScaler):


,Metric,Train,Test
0,MAE,713.55,708.55
1,RMSE,1037.76,1025.99
2,R²,0.5785,0.5859


#### d. ElasticNet Regression with StandardScaler

In [268]:
from sklearn.linear_model import ElasticNet

elastic_std = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=100000)
elastic_std.fit(X_train_std, y_train)

y_train_pred_elastic_std = elastic_std.predict(X_train_std)
y_test_pred_elastic_std = elastic_std.predict(X_test_std)

mae_train_elastic_std = np.mean(np.abs(y_train - y_train_pred_elastic_std))
mae_test_elastic_std = np.mean(np.abs(y_test - y_test_pred_elastic_std))
rmse_train_elastic_std = np.sqrt(np.mean((y_train - y_train_pred_elastic_std) ** 2))
rmse_test_elastic_std = np.sqrt(np.mean((y_test - y_test_pred_elastic_std) ** 2))
r2_train_elastic_std = my_r2_score(y_train, y_train_pred_elastic_std)
r2_test_elastic_std = my_r2_score(y_test, y_test_pred_elastic_std)

print_results_table(
    "ELASTICNET REGRESSION (StandardScaler)",
    mae_train_elastic_std, mae_test_elastic_std,
    rmse_train_elastic_std, rmse_test_elastic_std,
    r2_train_elastic_std, r2_test_elastic_std
)

ELASTICNET REGRESSION (StandardScaler):


,Metric,Train,Test
0,MAE,713.43,708.30
1,RMSE,1037.78,1025.90
2,R²,0.5785,0.5860


#### e. Summary Table - All Models with StandardScaler

In [269]:
models_std = [
    ("LinearRegression", mae_train_lr_std, mae_test_lr_std, rmse_train_lr_std, rmse_test_lr_std, r2_train_lr_std, r2_test_lr_std),
    ("Ridge", mae_train_ridge_std, mae_test_ridge_std, rmse_train_ridge_std, rmse_test_ridge_std, r2_train_ridge_std, r2_test_ridge_std),
    ("Lasso", mae_train_lasso_std, mae_test_lasso_std, rmse_train_lasso_std, rmse_test_lasso_std, r2_train_lasso_std, r2_test_lasso_std),
    ("ElasticNet", mae_train_elastic_std, mae_test_elastic_std, rmse_train_elastic_std, rmse_test_elastic_std, r2_train_elastic_std, r2_test_elastic_std)
]

create_summary_table(models_std, "SUMMARY - MODELS WITH STANDARD SCALER")

SUMMARY - MODELS WITH STANDARD SCALER


MAE     RMSE      R2
model            split                         
LinearRegression train  713.56  1037.76  0.5785
                 test   708.55  1025.99  0.5859
Ridge            train  713.55  1037.76  0.5785
                 test   708.55  1025.99  0.5859
Lasso            train  713.55  1037.76  0.5785
                 test   708.55  1025.99  0.5859
ElasticNet       train  713.43  1037.78  0.5785
                 test   708.30  1025.90  0.5860

### iii. Add all results to our dataframe with metrics on samples.

In [270]:

all_models_data = [
    ("my_LinearRegression (Standard)", mae_train_my, mae_test_my, rmse_train_my, rmse_test_my, r2_train_my, r2_test_my),
    ("sklearn_LinearRegression (Standard)", mae_train_sk, mae_test_sk, rmse_train_sk, rmse_test_sk, r2_train_sk, r2_test_sk),
    
    ("my_Ridge (Standard)", mae_train_ridge, mae_test_ridge, rmse_train_ridge, rmse_test_ridge, r2_train_ridge, r2_test_ridge),
    ("sklearn_Ridge (Standard)", mae_test_ridge_std, mae_test_ridge_std, rmse_train_ridge_std, rmse_test_ridge_std, r2_train_ridge_std, r2_test_ridge_std),
    
    ("my_Lasso (Standard)", mae_train_lasso, mae_test_lasso, rmse_train_lasso, rmse_test_lasso, r2_train_lasso, r2_test_lasso),
    ("sklearn_Lasso (Standard)", mae_train_lasso_std, mae_test_lasso_std, rmse_train_lasso_std, rmse_test_lasso_std, r2_train_lasso_std, r2_test_lasso_std),
    
    ("my_ElasticNet (Standard)", mae_train_elastic, mae_test_elastic, rmse_train_elastic, rmse_test_elastic, r2_train_elastic, r2_test_elastic),
    ("sklearn_ElasticNet (Standard)", mae_train_elastic_std, mae_test_elastic_std, rmse_train_elastic_std, rmse_test_elastic_std, r2_train_elastic_std, r2_test_elastic_std),
    
    ("sklearn_LinReg (MinMax)", mae_train_lr_mm, mae_test_lr_mm, rmse_train_lr_mm, rmse_test_lr_mm, r2_train_lr_mm, r2_test_lr_mm),
    ("sklearn_Ridge (MinMax)", mae_train_ridge_mm, mae_test_ridge_mm, rmse_train_ridge_mm, rmse_test_ridge_mm, r2_train_ridge_mm, r2_test_ridge_mm),
    ("sklearn_Lasso (MinMax)", mae_train_lasso_mm, mae_test_lasso_mm, rmse_train_lasso_mm, rmse_test_lasso_mm, r2_train_lasso_mm, r2_test_lasso_mm),
    ("sklearn_ElasticNet (MinMax)", mae_train_elastic_mm, mae_test_elastic_mm, rmse_train_elastic_mm, rmse_test_elastic_mm, r2_train_elastic_mm, r2_test_elastic_mm),
]

all_results = pd.DataFrame(
    np.nan,
    index=pd.MultiIndex.from_product(
        [[name for name, _, _, _, _, _, _ in all_models_data], ["train", "test"]],
        names=["model", "split"],
    ),
    columns=["MAE", "RMSE", "R2"],
)

for name, mae_tr, mae_te, rmse_tr, rmse_te, r2_tr, r2_te in all_models_data:
    all_results.loc[(name, "train"), "MAE"] = mae_tr
    all_results.loc[(name, "train"), "RMSE"] = rmse_tr
    all_results.loc[(name, "train"), "R2"] = r2_tr
    all_results.loc[(name, "test"), "MAE"] = mae_te
    all_results.loc[(name, "test"), "RMSE"] = rmse_te
    all_results.loc[(name, "test"), "R2"] = r2_te

for col in ["MAE", "RMSE"]:
    all_results[col] = all_results[col].map(lambda x: f"{x:.2f}")
all_results["R2"] = all_results["R2"].map(lambda x: f"{x:.4f}")

print("FINAL SUMMARY - ALL MODELS")
display(all_results)

FINAL SUMMARY - ALL MODELS


MAE     RMSE      R2
model                               split                         
my_LinearRegression (Standard)      train  713.56  1037.76  0.5785
                                    test   708.55  1025.99  0.5859
sklearn_LinearRegression (Standard) train  713.56  1037.76  0.5785
                                    test   708.55  1025.99  0.5859
my_Ridge (Standard)                 train  713.55  1037.76  0.5785
                                    test   708.55  1025.99  0.5859
sklearn_Ridge (Standard)            train  708.55  1037.76  0.5785
                                    test   708.55  1025.99  0.5859
my_Lasso (Standard)                 train  738.51  1054.91  0.5645
                                    test   734.00  1048.19  0.5678
sklearn_Lasso (Standard)            train  713.55  1037.76  0.5785
                                    test   708.55  1025.99  0.5859
my_ElasticNet (Standard)            train  718.85  1054.70  0.5646
                                    test   715.56  1042.95  0.5721
sklearn_ElasticNet (Standard)       train  713.43  1037.78  0.5785
                                    test   708.30  1025.90  0.5860
sklearn_LinReg (MinMax)             train  713.56  1037.76  0.5785
                                    test   708.55  1025.99  0.5859
sklearn_Ridge (MinMax)              train  713.71  1037.82  0.5785
                                    test   708.51  1026.06  0.5858
sklearn_Lasso (MinMax)              train  713.55  1037.76  0.5785
                                    test   708.54  1025.99  0.5859
sklearn_ElasticNet (MinMax)         train  774.60  1139.10  0.4922
                                    test   766.73  1132.46  0.4955

## 8. Overfit models

### ii. Apply StandardScaler to polynomial features

In [271]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split

X_base = df[['bathrooms', 'bedrooms']]
y_base = df['price']

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base, y_base, test_size=0.2, random_state=21
)

poly = PolynomialFeatures(degree=10, include_bias=False)
X_train_poly = poly.fit_transform(X_train_base)
X_test_poly = poly.transform(X_test_base)

print(f"Original features: {X_train_base.shape[1]}")
print(f"Polynomial features (degree=10): {X_train_poly.shape[1]}")

Original features: 2
Polynomial features (degree=10): 65


In [272]:
from sklearn.preprocessing import StandardScaler

scaler_poly = StandardScaler()
X_train_poly_scaled = scaler_poly.fit_transform(X_train_poly)
X_test_poly_scaled = scaler_poly.transform(X_test_poly)

print("StandardScaler applied to polynomial features")
print(f"X_train_poly_scaled shape: {X_train_poly_scaled.shape}")

StandardScaler applied to polynomial features
X_train_poly_scaled shape: (38703, 65)


### iii. Train and fit all our implemented algorithms — Linear Regression, Ridge, Lasso, and ElasticNet — on a set of polynomial features.

In [273]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet

lr_poly = LinearRegression()
lr_poly.fit(X_train_poly_scaled, y_train_base)

y_train_pred_poly = lr_poly.predict(X_train_poly_scaled)
y_test_pred_poly = lr_poly.predict(X_test_poly_scaled)

mae_train_poly = np.mean(np.abs(y_train_base - y_train_pred_poly))
mae_test_poly = np.mean(np.abs(y_test_base - y_test_pred_poly))
rmse_train_poly = np.sqrt(np.mean((y_train_base - y_train_pred_poly) ** 2))
rmse_test_poly = np.sqrt(np.mean((y_test_base - y_test_pred_poly) ** 2))
r2_train_poly = my_r2_score(y_train_base, y_train_pred_poly)
r2_test_poly = my_r2_score(y_test_base, y_test_pred_poly)

print_results_table(
    "LINEAR REGRESSION (Polynomial degree=10)",
    mae_train_poly, mae_test_poly,
    rmse_train_poly, rmse_test_poly,
    r2_train_poly, r2_test_poly
)

ridge_poly = Ridge(alpha=1.0)
ridge_poly.fit(X_train_poly_scaled, y_train_base)

y_train_pred_ridge_poly = ridge_poly.predict(X_train_poly_scaled)
y_test_pred_ridge_poly = ridge_poly.predict(X_test_poly_scaled)

mae_train_ridge_poly = np.mean(np.abs(y_train_base - y_train_pred_ridge_poly))
mae_test_ridge_poly = np.mean(np.abs(y_test_base - y_test_pred_ridge_poly))
rmse_train_ridge_poly = np.sqrt(np.mean((y_train_base - y_train_pred_ridge_poly) ** 2))
rmse_test_ridge_poly = np.sqrt(np.mean((y_test_base - y_test_pred_ridge_poly) ** 2))
r2_train_ridge_poly = my_r2_score(y_train_base, y_train_pred_ridge_poly)
r2_test_ridge_poly = my_r2_score(y_test_base, y_test_pred_ridge_poly)

print_results_table(
    "RIDGE REGRESSION (Polynomial degree=10)",
    mae_train_ridge_poly, mae_test_ridge_poly,
    rmse_train_ridge_poly, rmse_test_ridge_poly,
    r2_train_ridge_poly, r2_test_ridge_poly
)

lasso_poly = Lasso(alpha=1.0, max_iter=100000)
lasso_poly.fit(X_train_poly_scaled, y_train_base)

y_train_pred_lasso_poly = lasso_poly.predict(X_train_poly_scaled)
y_test_pred_lasso_poly = lasso_poly.predict(X_test_poly_scaled)

mae_train_lasso_poly = np.mean(np.abs(y_train_base - y_train_pred_lasso_poly))
mae_test_lasso_poly = np.mean(np.abs(y_test_base - y_test_pred_lasso_poly))
rmse_train_lasso_poly = np.sqrt(np.mean((y_train_base - y_train_pred_lasso_poly) ** 2))
rmse_test_lasso_poly = np.sqrt(np.mean((y_test_base - y_test_pred_lasso_poly) ** 2))
r2_train_lasso_poly = my_r2_score(y_train_base, y_train_pred_lasso_poly)
r2_test_lasso_poly = my_r2_score(y_test_base, y_test_pred_lasso_poly)

print_results_table(
    "LASSO REGRESSION (Polynomial degree=10)",
    mae_train_lasso_poly, mae_test_lasso_poly,
    rmse_train_lasso_poly, rmse_test_lasso_poly,
    r2_train_lasso_poly, r2_test_lasso_poly
)

from sklearn.linear_model import ElasticNet

elastic_poly = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=100000)
elastic_poly.fit(X_train_poly_scaled, y_train_base)

y_train_pred_elastic_poly = elastic_poly.predict(X_train_poly_scaled)
y_test_pred_elastic_poly = elastic_poly.predict(X_test_poly_scaled)

mae_train_elastic_poly = np.mean(np.abs(y_train_base - y_train_pred_elastic_poly))
mae_test_elastic_poly = np.mean(np.abs(y_test_base - y_test_pred_elastic_poly))
rmse_train_elastic_poly = np.sqrt(np.mean((y_train_base - y_train_pred_elastic_poly) ** 2))
rmse_test_elastic_poly = np.sqrt(np.mean((y_test_base - y_test_pred_elastic_poly) ** 2))
r2_train_elastic_poly = my_r2_score(y_train_base, y_train_pred_elastic_poly)
r2_test_elastic_poly = my_r2_score(y_test_base, y_test_pred_elastic_poly)

print_results_table(
    "ELASTICNET REGRESSION (Polynomial degree=10)",
    mae_train_elastic_poly, mae_test_elastic_poly,
    rmse_train_elastic_poly, rmse_test_elastic_poly,
    r2_train_elastic_poly, r2_test_elastic_poly
)

LINEAR REGRESSION (Polynomial degree=10):


,Metric,Train,Test
0,MAE,756.70,753.71
1,RMSE,1078.97,1073.36
2,R²,0.5444,0.5468


RIDGE REGRESSION (Polynomial degree=10):


,Metric,Train,Test
0,MAE,761.19,758.46
1,RMSE,1086.69,1080.90
2,R²,0.5378,0.5404


LASSO REGRESSION (Polynomial degree=10):


,Metric,Train,Test
0,MAE,764.59,762.13
1,RMSE,1093.43,1084.20
2,R²,0.5321,0.5376


ELASTICNET REGRESSION (Polynomial degree=10):


,Metric,Train,Test
0,MAE,788.78,784.60
1,RMSE,1127.76,1115.13
2,R²,0.5022,0.5108


### iv. Store the results in the result dataframe

In [274]:
all_models_final = [
    ("Linear Regression (Standard)", mae_test_sk, rmse_test_sk, r2_test_sk),
    ("My Linear Regression (Analytical)", mae_test_my, rmse_test_my, r2_test_my),
    
    ("Ridge (Standard)", mae_test_ridge_std, rmse_test_ridge_std, r2_test_ridge_std),
    ("My Ridge (Standard)", mae_test_ridge, rmse_test_ridge, r2_test_ridge),
    
    ("Lasso (Standard)", mae_test_lasso_std, rmse_test_lasso_std, r2_test_lasso_std),
    ("My Lasso (Standard)", mae_test_lasso, rmse_test_lasso, r2_test_lasso),
    
    ("ElasticNet (Standard)", mae_test_elastic_std, rmse_test_elastic_std, r2_test_elastic_std),
    ("My ElasticNet (Standard)", mae_test_elastic, rmse_test_elastic, r2_test_elastic),
    
    ("Linear Regression (MinMax)", mae_test_lr_mm, rmse_test_lr_mm, r2_test_lr_mm),
    ("Ridge (MinMax)", mae_test_ridge_mm, rmse_test_ridge_mm, r2_test_ridge_mm),
    ("Lasso (MinMax)", mae_test_lasso_mm, rmse_test_lasso_mm, r2_test_lasso_mm),
    ("ElasticNet (MinMax)", mae_test_elastic_mm, rmse_test_elastic_mm, r2_test_elastic_mm),
]

poly_models_to_add = [
    ("my_LinReg_poly_std", 3544.73, 3533.96, 3706.46, 3693.89, -4.3767, -4.3678),
    ("my_Lasso_poly_std", 3.176e92, 3.306e92, 2.684e93, 2.003e93, -2.819e180, -1.578e180),
    ("my_Ridge_poly_std", 6.973e88, 7.405e88, 5.765e89, 4.512e89, -1.301e173, -8.010e172),
    ("my_ElasticNet_poly_std", 2.218e91, 2.251e91, 1.175e92, 7.667e91, -5.405e177, -2.313e177),
]

for model in poly_models_to_add:
    name, mae_tr, mae_te, rmse_tr, rmse_te, r2_tr, r2_te = model
    all_models_final.append((name, mae_te, rmse_te, r2_te))

results_final_with_poly = pd.DataFrame(all_models_final, columns=["Model", "MAE", "RMSE", "R2"])

def format_value(x, is_metric=False):
    if isinstance(x, (int, float)):
        if abs(x) > 1e10 or (abs(x) < 1e-10 and x != 0):
            return f"{x:.2e}"
        else:
            return f"{x:.2f}"
    return str(x)

results_final_with_poly["MAE"] = results_final_with_poly["MAE"].apply(lambda x: format_value(x))
results_final_with_poly["RMSE"] = results_final_with_poly["RMSE"].apply(lambda x: format_value(x))
results_final_with_poly["R2"] = results_final_with_poly["R2"].apply(lambda x: f"{x:.4e}" if isinstance(x, float) and (x > 1e10 or x < -1e10) else f"{x:.4f}")

print("FINAL RESULTS - ALL MODELS (including Polynomial)")
display(results_final_with_poly)

FINAL RESULTS - ALL MODELS (including Polynomial)


,Model,MAE,RMSE,R2
0,Linear Regression (Standard),708.55,1025.99,0.5859
1,My Linear Regression (Analytical),708.55,1025.99,0.5859
2,Ridge (Standard),708.55,1025.99,0.5859
3,My Ridge (Standard),708.55,1025.99,0.5859
4,Lasso (Standard),708.55,1025.99,0.5859
5,My Lasso (Standard),734.00,1048.19,0.5678
6,ElasticNet (Standard),708.30,1025.90,0.5860
7,My ElasticNet (Standard),715.56,1042.95,0.5721
8,Linear Regression (MinMax),708.55,1025.99,0.5859
9,Ridge (MinMax),708.51,1026.06,0.5858


### v. Select the best model according to your opinion.

**Лучшая модель**: ElasticNet (Standard) — MAE = 708.30, R² = 0.5860

- *Самая низкая MAE на тестовых данных*
- *Самый высокий R²*
- *Не переобучена*

## 9. Native Models

### i. Calculate the mean and median metrics from the previous lesson and add the results to the final dataframe.

In [275]:
class MeanRegressor:
    def __init__(self):
        self.mean = None
    
    def fit(self, X, y):
        self.mean = np.mean(y)
        return self
    
    def predict(self, X):
        return np.full(X.shape[0], self.mean)


class MedianRegressor:
    def __init__(self):
        self.median = None
    
    def fit(self, X, y):
        self.median = np.median(y)
        return self
    
    def predict(self, X):
        return np.full(X.shape[0], self.median)

In [276]:
mean_model = MeanRegressor()
median_model = MedianRegressor()

mean_model.fit(X_train_scaled, y_train)
median_model.fit(X_train_scaled, y_train)

y_train_pred_mean = mean_model.predict(X_train_scaled)
y_test_pred_mean = mean_model.predict(X_test_scaled)

y_train_pred_median = median_model.predict(X_train_scaled)
y_test_pred_median = median_model.predict(X_test_scaled)

mae_train_mean = np.mean(np.abs(y_train - y_train_pred_mean))
mae_test_mean = np.mean(np.abs(y_test - y_test_pred_mean))
rmse_train_mean = np.sqrt(np.mean((y_train - y_train_pred_mean) ** 2))
rmse_test_mean = np.sqrt(np.mean((y_test - y_test_pred_mean) ** 2))
r2_train_mean = my_r2_score(y_train, y_train_pred_mean)
r2_test_mean = my_r2_score(y_test, y_test_pred_mean)

mae_train_median = np.mean(np.abs(y_train - y_train_pred_median))
mae_test_median = np.mean(np.abs(y_test - y_test_pred_median))
rmse_train_median = np.sqrt(np.mean((y_train - y_train_pred_median) ** 2))
rmse_test_median = np.sqrt(np.mean((y_test - y_test_pred_median) ** 2))
r2_train_median = my_r2_score(y_train, y_train_pred_median)
r2_test_median = my_r2_score(y_test, y_test_pred_median)

In [277]:
native_models_to_add = [
    ("Mean Regressor (Native)", mae_test_mean, rmse_test_mean, r2_test_mean),
    ("Median Regressor (Native)", mae_test_median, rmse_test_median, r2_test_median),
]

for model in native_models_to_add:
    name, mae, rmse, r2 = model
    all_models_final.append((name, mae, rmse, r2))

results_final_with_native = pd.DataFrame(all_models_final, columns=["Model", "MAE", "RMSE", "R2"])

def format_numeric(x):
    if isinstance(x, (int, float)):
        if abs(x) > 1e10:
            return f"{x:.2e}"
        elif abs(x) < 1e-10 and x != 0:
            return f"{x:.2e}"
        else:
            return f"{x:.2f}"
    return str(x)

results_final_with_native["MAE"] = results_final_with_native["MAE"].apply(format_numeric)
results_final_with_native["RMSE"] = results_final_with_native["RMSE"].apply(format_numeric)
results_final_with_native["R2"] = results_final_with_native["R2"].apply(lambda x: f"{x:.4f}" if isinstance(x, (int, float)) and abs(x) < 1e10 else f"{x:.2e}")

print("FINAL RESULTS - ALL MODELS (including Native Models)")
display(results_final_with_native)

FINAL RESULTS - ALL MODELS (including Native Models)


,Model,MAE,RMSE,R2
0,Linear Regression (Standard),708.55,1025.99,0.5859
1,My Linear Regression (Analytical),708.55,1025.99,0.5859
2,Ridge (Standard),708.55,1025.99,0.5859
3,My Ridge (Standard),708.55,1025.99,0.5859
4,Lasso (Standard),708.55,1025.99,0.5859
5,My Lasso (Standard),734.00,1048.19,0.5678
6,ElasticNet (Standard),708.30,1025.90,0.5860
7,My ElasticNet (Standard),715.56,1042.95,0.5721
8,Linear Regression (MinMax),708.55,1025.99,0.5859
9,Ridge (MinMax),708.51,1026.06,0.5858


## 10. Compare results

### i. Print your final tables

In [278]:
print("FINAL RESULTS - ALL MODELS")
display(results_final_with_native)

FINAL RESULTS - ALL MODELS


,Model,MAE,RMSE,R2
0,Linear Regression (Standard),708.55,1025.99,0.5859
1,My Linear Regression (Analytical),708.55,1025.99,0.5859
2,Ridge (Standard),708.55,1025.99,0.5859
3,My Ridge (Standard),708.55,1025.99,0.5859
4,Lasso (Standard),708.55,1025.99,0.5859
5,My Lasso (Standard),734.00,1048.19,0.5678
6,ElasticNet (Standard),708.30,1025.90,0.5860
7,My ElasticNet (Standard),715.56,1042.95,0.5721
8,Linear Regression (MinMax),708.55,1025.99,0.5859
9,Ridge (MinMax),708.51,1026.06,0.5858


### ii. What is the best model?

In [279]:
valid_models = results_final_with_native.iloc[:12]  
valid_models = pd.concat([valid_models, results_final_with_native.iloc[16:]]) 

def is_valid_r2(value):
    try:
        val = float(value)
        return val > -10 and val < 10
    except:
        return False

valid_models = results_final_with_native[results_final_with_native["R2"].apply(is_valid_r2)]
best_mae = valid_models.loc[valid_models["MAE"].astype(float).idxmin()]
best_r2 = valid_models.loc[valid_models["R2"].astype(float).idxmax()]

print(f"   {best_mae['Model']}")
print(f"   MAE = {best_mae['MAE']}")
print(f"   R² = {best_mae['R2']}")


   ElasticNet (Standard)
   MAE = 708.30
   R² = 0.5860


### iii. Which is the most stable model?

In [280]:
stability_data = []

stability_models = [
    ("Linear Regression (Standard)", r2_train_sk, r2_test_sk),
    ("Ridge (Standard)", r2_train_ridge_std, r2_test_ridge_std),
    ("Lasso (Standard)", r2_train_lasso_std, r2_test_lasso_std),
    ("ElasticNet (Standard)", r2_train_elastic_std, r2_test_elastic_std),
    ("My Linear Regression (Analytical)", r2_train_my, r2_test_my),
    ("Mean Regressor (Native)", r2_train_mean, r2_test_mean),
    ("Median Regressor (Native)", r2_train_median, r2_test_median),
]

for name, r2_tr, r2_te in stability_models:
    diff = abs(r2_tr - r2_te)
    stability_data.append({
        "Model": name,
        "R²_train": r2_tr,
        "R²_test": r2_te,
        "|diff|": diff
    })

stability_df = pd.DataFrame(stability_data)
stability_df["R²_train"] = stability_df["R²_train"].map(lambda x: f"{x:.4f}")
stability_df["R²_test"] = stability_df["R²_test"].map(lambda x: f"{x:.4f}")
stability_df["|diff|"] = stability_df["|diff|"].map(lambda x: f"{x:.6f}")

most_stable = stability_df.loc[stability_df["|diff|"].astype(float).idxmin()]

print(f"{most_stable['Model']}")

Mean Regressor (Native)
